## Reading images with the ground truth 

### Pre- processing the ImHANDWRTITNG dataset 1 : metadata creation

In [37]:
import os
import csv
import concurrent.futures
from collections import defaultdict

# Paths
base_folder = "/Users/krishnanand/Documents/Git/Data/IMHANDWRITING_1/"  # This should contain formsA-D, formsE-H, ...
lines_txt_path = "/Users/krishnanand/Documents/Git/Data/IMHANDWRITING_1/ascii/lines.txt"  # Adjust if needed
output_csv = "Handwriting_1_matadata.csv"

# Step 1: Load and parse lines.txt
form_lines = defaultdict(list)

with open(lines_txt_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.startswith("#") or not line.strip():
            continue
        parts = line.strip().split()
        line_id = parts[0]
        form_id = "-".join(line_id.split("-")[:2])
        text = " ".join(parts[8:]).replace("|", " ")
        form_lines[form_id].append(text)

# Step 2: Scan all folders for image paths
def scan_for_images(folder):
    img_paths = {}
    for root, _, files in os.walk(folder):
        for file in files:
            if file.endswith((".png", ".jpeg", ".jpg")):
                form_id = os.path.splitext(file)[0]
                img_paths[form_id] = os.path.join(root, file)
    return img_paths

# Use multithreading for scanning
form_image_paths = {}

with concurrent.futures.ThreadPoolExecutor() as executor:
    futures = [executor.submit(scan_for_images, os.path.join(base_folder, subfolder))
               for subfolder in os.listdir(base_folder)
               if subfolder.startswith("forms")]
    for future in concurrent.futures.as_completed(futures):
        form_image_paths.update(future.result())

# Step 3: Build dataset
dataset = []

for form_id, lines in form_lines.items():
    if form_id in form_image_paths:
        image_path = form_image_paths[form_id]
        text = " ".join(lines)
        dataset.append([form_id, image_path, text])

# Step 4: Write to CSV
with open(output_csv, "w", newline="", encoding="utf-8") as out_file:
    writer = csv.writer(out_file)
    writer.writerow(["form_id", "image_path", "transcription"])
    writer.writerows(dataset)

print(f"Dataset saved to: {output_csv}")

## Preprocessing image

In [31]:
import os
import cv2
import numpy as np

def resize_by_width_auto_scale(
    image_path,
    target_width=512,
    max_height=128,
    binarize=False,
    centered_pad=False
):
    """
    Resize image to fixed width, scale height accordingly.
    If height > max_height → downscale proportionally.
    If height < max_height → pad (bottom or center).
    Returns: normalized float32 image [1 x H x W]
    """
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(f"Image not found: {image_path}")

    orig_h, orig_w = img.shape
    scale = target_width / orig_w
    new_h = int(orig_h * scale)

    # Resize to fixed width
    img = cv2.resize(img, (target_width, new_h), interpolation=cv2.INTER_AREA)

    if new_h > max_height:
        # Downscale entire image to fit max_height
        scale = max_height / new_h
        new_w = int(target_width * scale)
        img = cv2.resize(img, (new_w, max_height), interpolation=cv2.INTER_AREA)

        # Pad right side if new width is smaller
        pad_img = np.ones((max_height, target_width), dtype=np.uint8) * 255
        pad_img[:, :new_w] = img
        img = pad_img
    else:
        # Pad bottom or center vertically
        pad_img = np.ones((max_height, target_width), dtype=np.uint8) * 255
        if centered_pad:
            top = (max_height - new_h) // 2
            pad_img[top:top + new_h, :] = img
        else:
            pad_img[:new_h, :] = img
        img = pad_img

    if binarize:
        img = cv2.adaptiveThreshold(
            img, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, 15, 10
        )

    img = img.astype(np.float32) / 255.0
    img = np.expand_dims(img, axis=0)  # [1, H, W]
    return img


def process_folder_by_width(
    input_folder,
    output_folder,
    target_width=512,
    max_height=128,
    binarize=False,
    centered_pad=False,
    out_ext=".png",
    verbose=True
):
    """
    Resize & pad all images in a folder using width-based scaling.
    Saves output to mirrored structure in output_folder.
    """
    os.makedirs(output_folder, exist_ok=True)

    for fname in os.listdir(input_folder):
        if not fname.lower().endswith((".png", ".jpg", ".jpeg")):
            continue
        try:
            input_path = os.path.join(input_folder, fname)
            output_name = os.path.splitext(fname)[0] + out_ext
            output_path = os.path.join(output_folder, output_name)

            img = resize_by_width_auto_scale(
                input_path,
                target_width=target_width,
                max_height=max_height,
                binarize=binarize,
                centered_pad=centered_pad
            )
            save_img = (img[0] * 255).astype(np.uint8)
            cv2.imwrite(output_path, save_img)

            if verbose:
                print(f"✅ Saved: {output_name}")
        except Exception as e:
            print(f"❌ Failed on {fname}: {e}")

In [32]:
process_folder_by_width(
    input_folder="/Users/krishnanand/Documents/Git/Data/IMHANDWRITING_1/formsA-D",
    output_folder="/Users/krishnanand/Documents/Git/Data/IMHANDWRITING_1/processed/",
    target_width=1024,
    max_height=128,
    binarize= False
)

In [33]:
process_folder_by_width(
    input_folder="/Users/krishnanand/Documents/Git/Data/IMHANDWRITING_1/formsE-H",
    output_folder="/Users/krishnanand/Documents/Git/Data/IMHANDWRITING_1/processed/",
    target_width=1024,
    max_height=128,
    binarize=False
)

In [34]:
process_folder_by_width(
    input_folder="/Users/krishnanand/Documents/Git/Data/IMHANDWRITING_1/formsI-Z",
    output_folder="/Users/krishnanand/Documents/Git/Data/IMHANDWRITING_1/processed/",
    target_width=1024,
    max_height=128,
    binarize=False
)

## Creating Dataframe

In [35]:
import pandas as pd
df = pd.read_csv("Handwriting_1_matadata.csv")

In [36]:
df

### Tokenization 

In [37]:
import pandas as pd

class OCRTokenizer:
    def __init__(self, text_column='transcription'):
        text_data = "".join(df[text_column].astype(str).tolist())

        self.text_column = text_column
        self.vocab = sorted(set(text_data))
        self.char2idx = {char: idx + 1 for idx, char in enumerate(self.vocab)}  # Start from 1
        self.char2idx['[BLANK]'] = 0  # CTC blank token

        self.idx2char = {idx: char for char, idx in self.char2idx.items()}

    def encode(self, text):
        return [self.char2idx[char] for char in text if char in self.char2idx]

    def decode(self, indices):
        return "".join([self.idx2char.get(idx, '') for idx in indices if idx > 0])

    def __len__(self):
        return len(self.char2idx)


def tokenize_dataset_csv(csv_path, tokenizer, text_column="transcription", save_as=None):
    """
    Tokenizes the entire dataset using the tokenizer.
    Adds 'encoded_label' column and optionally saves to disk.
    """
    df = pd.read_csv(csv_path)
    
    if text_column not in df.columns:
        raise ValueError(f"❌ Column '{text_column}' not found in CSV.")

    df['encoded_label'] = df[text_column].astype(str).apply(tokenizer.encode)

    if save_as:
        df.to_csv(save_as, index=False)
        print(f"✅ Tokenized CSV saved to: {save_as}")

    return df

In [38]:
tokenizer = OCRTokenizer(text_column="transcription")

tokenized_df = tokenize_dataset_csv(
    "Handwriting_1_matadata.csv",
    tokenizer,
    text_column="transcription",
    save_as="Handwriting_1_tokenized.csv"
)

### Train and validation set splits

In [39]:
import pandas as pd
import os
import random

def split_dataset_csv(
    csv_path,
    train_csv_path,
    test_csv_path,
    split_ratio=0.8,
    shuffle=True,
    seed=42
):
    """
    Split a labeled dataset CSV into train/test.
    """
    df = pd.read_csv(csv_path)

    if shuffle:
        df = df.sample(frac=1, random_state=seed).reset_index(drop=True)

    split_index = int(len(df) * split_ratio)
    train_df = df.iloc[:split_index]
    test_df = df.iloc[split_index:]

    train_df.to_csv(train_csv_path, index=False)
    test_df.to_csv(test_csv_path, index=False)

    print(f"Train: {len(train_df)} samples")
    print(f"Test:  {len(test_df)} samples")

In [40]:
split_dataset_csv(
    csv_path="/Users/krishnanand/Documents/Git/VisionSpeak/Handwriting_1_tokenized.csv",
    train_csv_path="/Users/krishnanand/Documents/Git/VisionSpeak/train.csv",
    test_csv_path="/Users/krishnanand/Documents/Git/VisionSpeak/test.csv",
    split_ratio=0.9
)

### Creating Dataloder

In [42]:
import torch
from torch.utils.data import Dataset, DataLoader
import cv2
import numpy as np
import pandas as pd
import ast

class OCRDataset(Dataset):
    def __init__(self, csv_path):
        self.df = pd.read_csv(csv_path)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row["image_path"]
        label_encoded = ast.literal_eval(row["encoded_label"])

        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise FileNotFoundError(f"Image not found: {img_path}")

        # Images are already resized/padded — no resize here
        img = img.astype(np.float32) / 255.0
        img = np.expand_dims(img, axis=0)  # [1, H, W]

        return {
            "image": torch.tensor(img, dtype=torch.float32),
            "label": torch.tensor(label_encoded, dtype=torch.long),
            "label_length": len(label_encoded),
            "img_path": img_path
        }

def ctc_collate_fn(batch):
    images = [item["image"] for item in batch]
    labels = [item["label"] for item in batch]

    image_batch = torch.stack(images)
    label_lengths = torch.tensor([len(label) for label in labels], dtype=torch.long)
    padded_labels = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=0)

    return {
        "images": image_batch,
        "labels": padded_labels,
        "label_lengths": label_lengths,
        "img_paths": [item["img_path"] for item in batch]
    }

# === Usage ===
train_csv_path = "/Users/krishnanand/Documents/Git/VisionSpeak/train.csv"
test_csv_path  = "/Users/krishnanand/Documents/Git/VisionSpeak/test.csv"

train_dataset = OCRDataset(train_csv_path)
test_dataset  = OCRDataset(test_csv_path)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, collate_fn=ctc_collate_fn)
test_loader  = DataLoader(test_dataset, batch_size=2, shuffle=False, collate_fn=ctc_collate_fn)

print("✅ DataLoaders ready!")
print("🔢 Train size:", len(train_dataset))
print("🔢 Test size:", len(test_dataset))

## Modelling

#### Baseline 1 

In [ ]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import torch
from PIL import Image

# === TrOCR Dataset ===
class TrOCRDataset(Dataset):
    def __init__(self, csv_path, processor):
        self.df = pd.read_csv(csv_path)
        self.processor = processor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['image_path']).convert("RGB")
        pixel_values = self.processor(images=img, return_tensors="pt").pixel_values.squeeze(0)
        
        return {
            'pixel_values': pixel_values,
            'text': row['transcription']  # fixed here
        }

# === Collate Function ===
def trocr_collate_fn(batch, processor):
    pixel_values = torch.stack([item['pixel_values'] for item in batch])
    texts = [item['text'] for item in batch]
    labels = processor.tokenizer(texts, padding="longest", return_tensors="pt").input_ids
    labels[labels == processor.tokenizer.pad_token_id] = -100  # For CrossEntropyLoss
    return {'pixel_values': pixel_values, 'labels': labels}

# === Load Pretrained Model and Processor ===
processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-handwritten")
model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-handwritten")
model.config.decoder_start_token_id = processor.tokenizer.bos_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
model.to(device)

# === DataLoader ===
train_dataset = TrOCRDataset("/Users/krishnanand/Documents/Git/VisionSpeak/train.csv", processor)
test_dataset = TrOCRDataset("/Users/krishnanand/Documents/Git/VisionSpeak/test.csv", processor)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True,
                          collate_fn=lambda x: trocr_collate_fn(x, processor))
test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False,
                         collate_fn=lambda x: trocr_collate_fn(x, processor))

# === Training Loop ===
from torch import nn
from transformers import AdamW

optimizer = AdamW(model.parameters(), lr=5e-5)
loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

for epoch in range(3):
    model.train()
    total_loss = 0
    for batch in train_loader:
        pixel_values = batch['pixel_values'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(pixel_values=pixel_values, labels=labels)
        loss = outputs.loss
        loss.backward()

        optimizer.step()
        optimizer.zero_grad()

        total_loss += loss.item()

    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in test_loader:
            pixel_values = batch['pixel_values'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(pixel_values=pixel_values, labels=labels)
            loss = outputs.loss
            val_loss += loss.item()

    print(f"📅 Epoch {epoch+1} — 🏋️ Train Loss: {total_loss / len(train_loader):.4f} | 🧪 Val Loss: {val_loss / len(test_loader):.4f}")

print("✅ TrOCR Baseline training complete!")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.48, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Config of the encoder: <class 'transformers.models.vit.modeling_vit.ViTModel'> is overwritten by shared encoder config: ViTConfig {
  "attention_probs_dropout_prob": 0.0,
  "encoder_stride": 16,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.0,
  "hidden_size": 768,
  "image_size": 384,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "model_type": "vit",
  "num_attention_heads": 12,
  "num_channels": 3,
  "num_hidden_layers": 12,
  "patch_size": 16,
  "qkv_bias": false,
  "torch_dtype": "float32",
  "transformers_version": "4.49.0"
}

Config of the decoder: <class 'transformers.models.trocr.modeling_trocr.TrOCRForCausalLM'> i

📅 Epoch 1 — 🏋️ Train Loss: 6.0536
📅 Epoch 2 — 🏋️ Train Loss: 5.0637
📅 Epoch 3 — 🏋️ Train Loss: 4.6171
✅ TrOCR Baseline training complete!


## Language Detection

In [ ]:
import fasttext
import torch

# Detect the best available device
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

# Ensure correct path to model file
model_path = "/Users/krishnanand/Documents/Git/Dependencies/lid.176.bin"  # Update this path if necessary
model = fasttext.load_model(model_path)

def detect_language(text):
    """Detect language of given text using FastText."""
    predictions = model.predict(text)
    lang_code = predictions[0][0].replace('__label__', '')
    return lang_code

text = "Bonjour, comment allez-vous?"
detected_lang = detect_language(text)
print(f"Detected Language: {detected_lang}")

## Summarization  

In [ ]:
from transformers import pipeline
import torch

device = 0 if torch.cuda.is_available() else (0 if torch.backends.mps.is_available() else -1)

# Load summarization pipeline (T5-Base)
summarizer = pipeline("summarization", model="t5-base", device=device)

# Example input text
text = """
Artificial intelligence (AI) is the simulation of human intelligence processes by machines, especially computer systems. 
These processes include learning, reasoning, and self-correction. AI is one of the most exciting and disruptive technologies 
of our time, influencing everything from healthcare to transportation.
"""

# Generate summary
summary = summarizer(text, max_length=100, min_length=30, do_sample=False)
summary_text = summary[0]['summary_text']
print("Summary in English:", summary_text)

# Load translation pipeline with a better model
translator = pipeline("translation", model="Helsinki-NLP/opus-mt-en-es", device=device)

# Translate the summary
translated_summary = translator(summary_text)
print("Translated Summary in Spanish:", translated_summary[0]['translation_text'])

## Translator

In [ ]:
from transformers import pipeline
import torch

device = 0 if torch.cuda.is_available() else (0 if torch.backends.mps.is_available() else -1)

# Load the NLLB model for French to Hindi translation
translator = pipeline("translation", model="facebook/nllb-200-distilled-600M", device=device)

# Example French text
text = "Bonjour, comment allez-vous aujourd'hui?"

# Translate the text from French to English
translated_text = translator(text, src_lang="fra_Latn", tgt_lang="eng_Latn")
english_translation = translated_text[0]['translation_text']
print("Translated to English:", english_translation)

# Now, translate from English to Hindi
translated_to_hindi = translator(english_translation, src_lang="eng_Latn", tgt_lang="hin_Deva")
print("Translated to Hindi:", translated_to_hindi[0]['translation_text'])

## TTS

In [ ]:
from TTS.api import TTS
import torch

# Detect best available device (CUDA for NVIDIA, MPS for Apple Silicon, CPU as fallback)
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

# Load a single-language English TTS model and move it to the selected device
tts = TTS("tts_models/en/ljspeech/tacotron2-DDC").to(device)

# Text to convert
text = ("The error 'Model is not multi-lingual but language is provided' means that "
        "the model does not support multiple languages. This model can only generate speech in English, "
        "so specifying language (French) is invalid.")

# Convert text to speech and save as an audio file
tts.tts_to_file(text=text, file_path="/Users/krishnanand/Documents/Git/Tests/test123.wav")

print("Speech synthesis complete! Output saved as 'test123.wav'.")

In [ ]:
pip install colabcode

In [ ]:
%pip install colabcode